# PyFlink 실시간 데이터 처리 튜토리얼

이 노트북은 Apache Flink의 Python API인 PyFlink를 사용하여 실시간 데이터 처리를 학습합니다.

## 목차

1. **환경 설정**
   - PyFlink 설치
   - Flink UI 모니터링
2. **DataStream API**
   - 기본 연산 (map, filter, key_by, reduce)
   - Word Count 예제
   - JSON 데이터 처리
   - 상태 관리 (State)
   - 윈도우 연산
   - 이벤트 시간과 타이머
3. **Table API**
   - 기본 연산
   - UDF (User Defined Functions)
   - 윈도우 연산
   - SQL 연산
   - Pandas 연동
4. **DataStream과 Table API 혼합 사용**
5. **커넥터 (Connectors)**

---

**참고 문서**: [Flink Python API 공식 문서](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/overview/)

## 1. 환경 설정

### 필수 요구사항
- Java 11
- Python 3.9, 3.10, 3.11, or 3.12

### PyFlink 설치
```bash
pip install apache-flink
```

In [1]:
# PyFlink 설치 확인
import pyflink
print(f"PyFlink 버전: {pyflink.__version__}")

PyFlink 버전: 2.2.0


In [2]:
# 공통 import
import json
from typing import Iterable

# DataStream API imports
from pyflink.common import Types, WatermarkStrategy, Time, Encoder, Row, Duration
from pyflink.common.time import Instant
from pyflink.common.watermark_strategy import TimestampAssigner
from pyflink.datastream import StreamExecutionEnvironment, RuntimeExecutionMode
from pyflink.datastream.functions import KeyedProcessFunction, RuntimeContext
from pyflink.datastream.state import ValueStateDescriptor, StateTtlConfig

# Table API imports
from pyflink.table import (
    TableEnvironment, StreamTableEnvironment, EnvironmentSettings,
    TableDescriptor, Schema, DataTypes, FormatDescriptor
)
from pyflink.table.expressions import col, lit, row_interval, CURRENT_ROW
from pyflink.table.udf import udf, udtf, udaf
from pyflink.table.window import Tumble, Slide, Session, Over

print("모든 모듈이 성공적으로 import 되었습니다!")

모든 모듈이 성공적으로 import 되었습니다!


### 1.2 Flink UI 모니터링

Flink 작업을 실행하면 로컬 대시보드(기본값: http://localhost:8081)를 통해 작업의 진행 상황을 모니터링할 수 있습니다. 주요 확인 사항은 다음과 같습니다.

1. **Jobs**: 현재 실행 중인 Job과 완료된 Job의 전체적인 상태(Running, Finished, Failed 등)를 확인합니다.
2. **Visualized Graph**: 작성한 코드가 어떻게 최적화되어 실행 파이프라인으로 구성되었는지 시각화된 그래프로 확인합니다. (Operator Chaining 등 확인 가능)
3. **Task Metrics**: 각 연산자(Operator)별로 처리된 데이터 개수(Records Sent/Received), 처리량(Throughput), 지연 시간(Latency)을 확인합니다.
4. **Exceptions**: 작업이 실패한 경우 메인 대시보드나 TaskManager 로그에서 원인이 되는 Stacktrace를 상세히 확인할 수 있습니다.
5. **Watermarks (윈도우 연산 시)**: 이벤트 시간 기반 처리를 할 때 Watermark가 제대로 갱신되고 있는지 확인하여 데이터 지연이나 누락 여부를 판단할 수 있습니다.
6. **Checkpoints**: 상태 관리(State Management)를 사용하는 경우, 체크포인트가 정상적으로 수행되고 있는지와 상태(State)의 크기 및 이력을 모니터링합니다.

---
## 2. DataStream API

DataStream API는 Flink의 저수준 API로, 스트림 처리의 핵심 구성 요소인 **상태(State)**와 **시간(Time)**에 대한 세밀한 제어가 가능합니다.

### 핵심 개념
- **StreamExecutionEnvironment**: Flink 프로그램의 진입점
- **DataStream**: 데이터 스트림을 나타내는 기본 추상화
- **Transformation**: map, filter, key_by, reduce 등의 연산

### 2.1 기본 환경 설정

In [4]:
# 로컬 미니 클러스터 대신 원격 클러스터에 연결
env = StreamExecutionEnvironment.get_execution_environment()

# 병렬성 설정 (로컬 테스트에서는 1로 설정)
env.set_parallelism(1)

print("StreamExecutionEnvironment 생성 완료")
print(f"병렬성: {env.get_parallelism()}")

StreamExecutionEnvironment 생성 완료
병렬성: 1


### 2.2 Word Count - DataStream API의 "Hello World"

Word Count는 분산 처리 시스템의 대표적인 예제입니다. 
이 예제를 통해 `flat_map`, `map`, `key_by`, `reduce` 연산을 학습합니다.

In [5]:
# 예제 데이터 (셰익스피어의 햄릿 일부)
word_count_data = [
    "To be, or not to be,--that is the question:--",
    "Whether 'tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune",
    "Or to take arms against a sea of troubles,"
]

print("입력 데이터:")
for line in word_count_data:
    print(f"  {line}")

입력 데이터:
  To be, or not to be,--that is the question:--
  Whether 'tis nobler in the mind to suffer
  The slings and arrows of outrageous fortune
  Or to take arms against a sea of troubles,


In [6]:
# Word Count 구현
env = StreamExecutionEnvironment.get_execution_environment()
env.set_runtime_mode(RuntimeExecutionMode.BATCH)  # 배치 모드 설정
env.set_parallelism(1)

# 소스: 컬렉션에서 데이터 읽기
ds = env.from_collection(word_count_data)
ds

In [7]:
# 단어 분리 함수
def split(line):
    yield from line.split()

# 변환 파이프라인:
# 1. flat_map: 각 라인을 단어로 분리
# 2. map: 각 단어를 (단어, 1) 튜플로 변환
# 3. key_by: 단어별로 그룹화
# 4. reduce: 같은 단어의 카운트를 합산
result = ds.flat_map(split) \
           .map(lambda word: (word, 1), 
                output_type=Types.TUPLE([Types.STRING(), Types.INT()])) \
           .key_by(lambda x: x[0]) \
           .reduce(lambda a, b: (a[0], a[1] + b[1]))

# 결과 출력
result.print()

# 실행
env.execute("Word Count Example") # Word Count Example이라는 이름의 job으로 실행

(a,1)
(Or,1)
(To,1)
(in,1)
(is,1)
(of,2)
(or,1)
(to,3)
(The,1)
(and,1)
(be,,1)
(not,1)
(sea,1)
(the,2)
('tis,1)
(arms,1)
(mind,1)
(take,1)
(arrows,1)
(nobler,1)
(slings,1)
(suffer,1)
(Whether,1)
(against,1)
(fortune,1)
(be,--that,1)
(troubles,,1)
(outrageous,1)
(question:--,1)


#### 🔥 Flink 클러스터에 Job 제출하기

위 코드는 **로컬 미니 클러스터**에서 실행됩니다. Flink UI(http://localhost:8081)에서 Job을 확인하려면 `flink run` 명령어로 클러스터에 제출해야 합니다.

**PyFlink의 특징**: Java와 달리 `create_remote_execution_environment()`가 없어서, 노트북에서 직접 원격 클러스터에 연결할 수 없습니다.

In [ ]:
%%writefile /opt/flink/src/wordcount_job.py
# Flink 클러스터에 제출할 Word Count Job
from pyflink.datastream import StreamExecutionEnvironment, RuntimeExecutionMode
from pyflink.common import Types

env = StreamExecutionEnvironment.get_execution_environment()
env.set_runtime_mode(RuntimeExecutionMode.BATCH)
env.set_parallelism(1)

word_count_data = [
    "To be, or not to be,--that is the question:--",
    "Whether 'tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune",
    "Or to take arms against a sea of troubles,"
]

def split(line):
    yield from line.split()

ds = env.from_collection(word_count_data)
result = ds.flat_map(split) \
           .map(lambda word: (word, 1), output_type=Types.TUPLE([Types.STRING(), Types.INT()])) \
           .key_by(lambda x: x[0]) \
           .reduce(lambda a, b: (a[0], a[1] + b[1]))

result.print()
env.execute("Word Count Job - Cluster")

In [ ]:
# Flink 클러스터에 Job 제출 (Flink UI에서 확인 가능!)
!flink run -py /opt/flink/src/wordcount_job.py

### 2.3 기본 연산: map, filter, key_by, sum

DataStream API의 핵심 변환 연산들을 살펴봅니다.

In [ ]:
# 예제 데이터: (id, JSON 문자열)
sample_data = [
    (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
    (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
    (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
    (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
]

print("입력 데이터:")
for item in sample_data:
    print(f"  ID: {item[0]}, Data: {item[1][:50]}...")

In [ ]:
# map 연산: 각 레코드를 변환
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

# map: tel 값을 1 증가
def update_tel(data):
    json_data = json.loads(data.info)
    json_data['tel'] += 1
    return data.id, json.dumps(json_data)

print("=== map 연산 결과 (tel + 1) ===")
ds.map(update_tel).print()
env.execute("Map Example")

In [ ]:
# filter 연산: 조건에 맞는 레코드만 선택
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

def update_tel(data):
    json_data = json.loads(data.info)
    json_data['tel'] += 1
    return data.id, json.dumps(json_data)

print("=== filter 연산 결과 (id == 1인 것만) ===")
ds.filter(lambda data: data.id == 1).map(update_tel).print()
env.execute("Filter Example")

In [ ]:
# key_by + sum: 그룹화 후 집계
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=sample_data,
    type_info=Types.ROW_NAMED(["id", "info"], [Types.INT(), Types.STRING()])
)

print("=== key_by + sum 연산 결과 (국가별 tel 합계) ===")
# (country, tel) 형태로 변환 후 국가별 합계
ds.map(lambda data: (json.loads(data.info)['addr']['country'], 
                     json.loads(data.info)['tel'])) \
  .key_by(lambda x: x[0]) \
  .sum(1) \
  .print()
env.execute("KeyBy Sum Example")

### 2.4 JSON 데이터 처리

실무에서 자주 사용되는 JSON 데이터 처리 패턴을 살펴봅니다.

In [ ]:
# JSON 데이터 처리 예제
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ]
)

def update_tel(data):
    """JSON 파싱 후 tel 값 증가"""
    json_data = json.loads(data[1])
    json_data['tel'] += 1
    return data[0], json_data  # dict 형태로 반환

def filter_by_country(data):
    """중국 데이터만 필터링 (이미 파싱된 dict 사용)"""
    return "China" in data[1]['addr']['country']

print("=== JSON 처리: China 데이터만 필터링 ===")
ds.map(update_tel).filter(filter_by_country).print()
env.execute("JSON Processing Example")

### 2.5 상태 관리 (State Management)

Flink의 핵심 기능 중 하나인 **상태 관리**를 살펴봅니다.
- `ValueState`: 단일 값 저장
- `StateTtlConfig`: 상태의 TTL(Time-To-Live) 설정

In [ ]:
# 상태 관리 예제: 사용자별 금액 합계
class SumFunction(KeyedProcessFunction):
    """사용자별 금액을 누적 합산하는 함수"""
    
    def __init__(self):
        self.state = None
    
    def open(self, runtime_context: RuntimeContext):
        # 상태 디스크립터 생성
        state_descriptor = ValueStateDescriptor("sum_state", Types.FLOAT())
        
        # TTL 설정: 1초 후 만료
        state_ttl_config = StateTtlConfig \
            .new_builder(Time.seconds(1)) \
            .set_update_type(StateTtlConfig.UpdateType.OnReadAndWrite) \
            .disable_cleanup_in_background() \
            .build()
        state_descriptor.enable_time_to_live(state_ttl_config)
        
        # 상태 초기화
        self.state = runtime_context.get_state(state_descriptor)
    
    def process_element(self, value, ctx: 'KeyedProcessFunction.Context'):
        # 현재 상태 값 조회
        current = self.state.value()
        if current is None:
            current = 0
        
        # 상태 업데이트
        current += value[1]
        self.state.update(current)
        
        # 결과 출력
        yield value[0], current

# 테스트 데이터
print("입력 데이터:")
state_data = [
    ('Alice', 110.1),
    ('Bob', 30.2),
    ('Alice', 20.0),
    ('Bob', 53.1),
    ('Alice', 13.1),
    ('Bob', 3.1),
    ('Bob', 16.1),
    ('Alice', 20.1)
]
for item in state_data:
    print(f"  {item}")

In [ ]:
# 상태 관리 실행
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=state_data,
    type_info=Types.TUPLE([Types.STRING(), Types.FLOAT()])
)

print("=== 사용자별 누적 합계 ===")
ds.key_by(lambda value: value[0]) \
  .process(SumFunction()) \
  .print()

env.execute("State Access Demo")

### 2.6 윈도우 연산 (Window Operations)

스트림 처리에서 윈도우는 무한 데이터 스트림을 유한한 청크로 나누어 처리하는 핵심 개념입니다.

#### 윈도우 유형
1. **Tumbling Window**: 고정 크기, 겹치지 않음
2. **Sliding Window**: 고정 크기, 겹칠 수 있음
3. **Session Window**: 활동 간격 기반, 동적 크기
4. **Count Window**: 개수 기반

In [ ]:
# 윈도우 예제용 데이터
# (key, timestamp_ms)
window_data = [
    ('hi', 1), ('hi', 2), ('hi', 3), ('hi', 4), 
    ('hi', 5), ('hi', 8), ('hi', 9), ('hi', 15)
]

print("윈도우 테스트 데이터:")
print("  Key: hi")
print("  Timestamps (ms):", [t[1] for t in window_data])
print()
print("타임라인:")
print("  0   5   10   15   20")
print("  |---|---|----|----|")
print("  1234 5  89   15")

In [ ]:
# Timestamp Assigner 정의
class MyTimestampAssigner(TimestampAssigner):
    """튜플의 두 번째 요소를 타임스탬프로 사용"""
    def extract_timestamp(self, value, record_timestamp) -> int:
        return int(value[1])

#### 2.6.1 Tumbling Time Window (고정 크기 윈도우)

In [ ]:
from pyflink.datastream import ProcessWindowFunction
from pyflink.datastream.window import TumblingEventTimeWindows, TimeWindow

class CountWindowProcessFunction(ProcessWindowFunction[tuple, tuple, str, TimeWindow]):
    """윈도우 내 요소 개수를 세는 함수"""
    def process(self, key: str, context: ProcessWindowFunction.Context[TimeWindow], 
                elements: Iterable[tuple]) -> Iterable[tuple]:
        count = len([e for e in elements])
        return [(key, context.window().start, context.window().end, count)]

# Tumbling Window: 5ms 크기
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    window_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

print("=== Tumbling Window (5ms) ===")
print("윈도우: [0,5), [5,10), [10,15), [15,20)")
print()

ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(TumblingEventTimeWindows.of(Time.milliseconds(5))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Tumbling Window Example")

#### 2.6.2 Sliding Time Window (슬라이딩 윈도우)

In [ ]:
from pyflink.datastream.window import SlidingEventTimeWindows

# Sliding Window: 크기 5ms, 슬라이드 2ms
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    window_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

print("=== Sliding Window (크기: 5ms, 슬라이드: 2ms) ===")
print("윈도우가 겹칠 수 있음!")
print()

ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(SlidingEventTimeWindows.of(Time.milliseconds(5), Time.milliseconds(2))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Sliding Window Example")

#### 2.6.3 Session Window (세션 윈도우)

In [ ]:
from pyflink.datastream.window import EventTimeSessionWindows

# Session Window: 5ms 간격
session_data = [
    ('hi', 1), ('hi', 2), ('hi', 3), ('hi', 4),  # 세션 1
    ('hi', 8), ('hi', 9),  # 세션 2 (5ms 이상 간격)
    ('hi', 15)  # 세션 3
]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    session_data,
    type_info=Types.TUPLE([Types.STRING(), Types.INT()])
)

watermark_strategy = WatermarkStrategy.for_monotonous_timestamps() \
    .with_timestamp_assigner(MyTimestampAssigner())

print("=== Session Window (gap: 5ms) ===")
print("데이터:", [t[1] for t in session_data])
print("예상 세션: [1-4], [8-9], [15]")
print()

ds.assign_timestamps_and_watermarks(watermark_strategy) \
  .key_by(lambda x: x[0], key_type=Types.STRING()) \
  .window(EventTimeSessionWindows.with_gap(Time.milliseconds(5))) \
  .process(CountWindowProcessFunction(),
           Types.TUPLE([Types.STRING(), Types.INT(), Types.INT(), Types.INT()])) \
  .print()

env.execute("Session Window Example")

#### 2.6.4 Count Window (개수 기반 윈도우)

In [ ]:
from pyflink.datastream import WindowFunction
from pyflink.datastream.window import CountWindow

class SumWindowFunction(WindowFunction[tuple, tuple, str, CountWindow]):
    """윈도우 내 값의 합계를 계산"""
    def apply(self, key: str, window: CountWindow, inputs: Iterable[tuple]):
        total = sum(i[0] for i in inputs)
        return [(key, total)]

# Count Window: 2개씩 그룹화
count_data = [
    (1, 'hi'), (2, 'hello'), (3, 'hi'), (4, 'hello'), 
    (5, 'hi'), (6, 'hello'), (6, 'hello')
]

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    count_data,
    type_info=Types.TUPLE([Types.INT(), Types.STRING()])
)

print("=== Count Window (2개씩) ===")
print("입력:", count_data)
print("hi: [1, 3] -> 4, [5] -> 대기")
print("hello: [2, 4] -> 6, [6, 6] -> 12")
print()

ds.key_by(lambda x: x[1], key_type=Types.STRING()) \
  .count_window(2) \
  .apply(SumWindowFunction(), Types.TUPLE([Types.STRING(), Types.INT()])) \
  .print()

env.execute("Count Window Example")

### 2.7 이벤트 시간과 타이머

Flink는 이벤트 시간 기반 처리를 위한 타이머 기능을 제공합니다.

In [ ]:
# 이벤트 타이머 예제
class TimerSumFunction(KeyedProcessFunction):
    """타이머를 사용한 지연 출력 함수"""
    
    def __init__(self):
        self.state = None
    
    def open(self, runtime_context: RuntimeContext):
        state_descriptor = ValueStateDescriptor("state", Types.FLOAT())
        state_ttl_config = StateTtlConfig \
            .new_builder(Time.seconds(1)) \
            .set_update_type(StateTtlConfig.UpdateType.OnReadAndWrite) \
            .disable_cleanup_in_background() \
            .build()
        state_descriptor.enable_time_to_live(state_ttl_config)
        self.state = runtime_context.get_state(state_descriptor)
    
    def process_element(self, value, ctx: 'KeyedProcessFunction.Context'):
        # 상태 업데이트
        current = self.state.value() or 0
        current += value[2]
        self.state.update(current)
        
        # 2초 후에 타이머 발동 등록
        ctx.timer_service().register_event_time_timer(ctx.timestamp() + 2000)
    
    def on_timer(self, timestamp: int, ctx: 'KeyedProcessFunction.OnTimerContext'):
        """타이머 발동 시 호출"""
        yield ctx.get_current_key(), self.state.value()

class EventTimestampAssigner(TimestampAssigner):
    def extract_timestamp(self, value, record_timestamp: int) -> int:
        return int(value[0])

# 테스트 데이터: (timestamp_ms, name, amount)
timer_data = [
    (1000, 'Alice', 110.1),
    (4000, 'Bob', 30.2),
    (3000, 'Alice', 20.0),
    (2000, 'Bob', 53.1),
    (5000, 'Alice', 13.1),
    (3000, 'Bob', 3.1),
    (7000, 'Bob', 16.1),
    (10000, 'Alice', 20.1)
]

print("타이머 테스트 데이터:")
print("  각 이벤트 처리 후 2초 뒤에 타이머 발동")
print("  타이머 발동 시 현재까지의 합계 출력")

In [ ]:
# 이벤트 타이머 실행
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)

ds = env.from_collection(
    collection=timer_data,
    type_info=Types.TUPLE([Types.LONG(), Types.STRING(), Types.FLOAT()])
)

print("=== Event Time Timer 결과 ===")
ds.assign_timestamps_and_watermarks(
    WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(2))
                     .with_timestamp_assigner(EventTimestampAssigner())
) \
  .key_by(lambda value: value[1]) \
  .process(TimerSumFunction()) \
  .print()

env.execute("Event Timer Example")

---
## 3. Table API

Table API는 SQL과 유사한 고수준 API로, 관계형 데이터 처리를 지원합니다.

### 특징
- SQL과 유사한 문법
- 자동 최적화
- 스트리밍과 배치 모두 지원
- UDF 지원

### 3.1 기본 환경 설정

In [ ]:
# Table 환경 생성
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

print("TableEnvironment 생성 완료")

### 3.2 Word Count - Table API

In [ ]:
# Table API Word Count
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

word_count_data = [
    "To be, or not to be,--that is the question:--",
    "Whether 'tis nobler in the mind to suffer",
    "The slings and arrows of outrageous fortune"
]

# 소스 테이블 생성
tab = t_env.from_elements(
    map(lambda i: (i,), word_count_data),
    DataTypes.ROW([DataTypes.FIELD('line', DataTypes.STRING())])
)

# Sink 테이블 생성 (print connector)
t_env.create_temporary_table(
    'sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('word', DataTypes.STRING())
                           .column('count', DataTypes.BIGINT())
                           .build())
                   .build()
)

# UDTF (User Defined Table Function): 한 행을 여러 행으로 확장
@udtf(result_types=[DataTypes.STRING()])
def split(line: Row):
    for word in line[0].split():
        yield Row(word)

print("=== Table API Word Count ===")
# 파이프라인: flat_map -> group_by -> count
tab.flat_map(split).alias('word') \
   .group_by(col('word')) \
   .select(col('word'), lit(1).count) \
   .execute_insert('sink') \
   .wait()

### 3.3 기본 연산: 컬럼 조작, 필터, 집계, 조인

In [ ]:
# Table API 기본 연산
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

# 소스 테이블
table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

# JSON 필드 추출
table = table.add_columns(
    col('data').json_value('$.name', DataTypes.STRING()).alias('name'),
    col('data').json_value('$.tel', DataTypes.STRING()).alias('tel'),
    col('data').json_value('$.addr.country', DataTypes.STRING()).alias('country')
).drop_columns(col('data'))

print("=== JSON 필드 추출 결과 ===")
table.execute().print()

In [ ]:
# limit
print("=== limit(3) 결과 ===")
table.limit(3).execute().print()

In [ ]:
# filter
print("=== filter (id != 3) 결과 ===")
table.filter(col('id') != 3).execute().print()

In [ ]:
# group by + aggregation
print("=== 국가별 집계 (count, max) ===")
table.group_by(col('country')) \
     .select(
         col('country'), 
         col('id').count.alias('cnt'), 
         col('tel').cast(DataTypes.BIGINT()).max.alias('max_tel')
     ) \
     .execute().print()

In [ ]:
# distinct
print("=== distinct country ===")
table.select(col('country')).distinct().execute().print()

In [ ]:
# join
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

table = table.add_columns(
    col('data').json_value('$.name', DataTypes.STRING()).alias('name'),
    col('data').json_value('$.tel', DataTypes.STRING()).alias('tel'),
    col('data').json_value('$.addr.country', DataTypes.STRING()).alias('country')
).drop_columns(col('data'))

right_table = t_env.from_elements(
    elements=[(1, 18), (2, 30), (3, 25), (4, 10)],
    schema=['id', 'age']
)

print("=== JOIN 결과 ===")
table.join(
    right_table.rename_columns(col('id').alias('r_id')), 
    col('id') == col('r_id')
).execute().print()

### 3.4 UDF (User Defined Functions)

In [ ]:
# UDF 예제: JSON 업데이트
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}')
    ],
    schema=['id', 'data']
)

# Sink 테이블
t_env.create_temporary_table(
    'sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('id', DataTypes.BIGINT())
                           .column('data', DataTypes.STRING())
                           .build())
                   .build()
)

# UDF: tel 값 증가
@udf(result_type=DataTypes.STRING())
def update_tel(data):
    json_data = json.loads(data)
    json_data['tel'] += 1
    return json.dumps(json_data)

print("=== UDF로 JSON 업데이트 (tel + 1) ===")
table.select(col('id'), update_tel(col('data'))) \
     .execute_insert('sink') \
     .wait()

### 3.5 SQL 연산

In [ ]:
# SQL 쿼리 사용
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[
        (1, '{"name": "Flink", "tel": 123, "addr": {"country": "Germany", "city": "Berlin"}}'),
        (2, '{"name": "hello", "tel": 135, "addr": {"country": "China", "city": "Shanghai"}}'),
        (3, '{"name": "world", "tel": 124, "addr": {"country": "USA", "city": "NewYork"}}'),
        (4, '{"name": "PyFlink", "tel": 32, "addr": {"country": "China", "city": "Hangzhou"}}')
    ],
    schema=['id', 'data']
)

print("=== SQL Query 결과 ===")
t_env.sql_query(f"SELECT * FROM {table}").execute().print()

In [ ]:
# SQL에서 UDTF 사용
@udtf(result_types=[DataTypes.STRING(), DataTypes.INT(), DataTypes.STRING()])
def parse_data(data: str):
    json_data = json.loads(data)
    yield json_data['name'], json_data['tel'], json_data['addr']['country']

t_env.create_temporary_function('parse_data', parse_data)

print("=== SQL + UDTF (LATERAL TABLE) ===")
t_env.execute_sql(f"""
    SELECT id, name, tel, country
    FROM {table}, LATERAL TABLE(parse_data(`data`)) t(name, tel, country)
""").print()

### 3.6 Table API 윈도우 연산

In [ ]:
# Table API 윈도우용 데이터 준비
from pyflink.common.time import Instant

window_table_data = [
    (Instant.of_epoch_milli(1000), 'Alice', 110.1),
    (Instant.of_epoch_milli(4000), 'Bob', 30.2),
    (Instant.of_epoch_milli(3000), 'Alice', 20.0),
    (Instant.of_epoch_milli(2000), 'Bob', 53.1),
    (Instant.of_epoch_milli(5000), 'Alice', 13.1),
    (Instant.of_epoch_milli(3000), 'Bob', 3.1),
    (Instant.of_epoch_milli(7000), 'Bob', 16.1),
    (Instant.of_epoch_milli(10000), 'Alice', 20.1)
]

print("윈도우 테스트 데이터: (timestamp, name, price)")
for item in window_table_data:
    print(f"  {item}")

In [ ]:
# Tumble Window (Table API)
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

# Watermark 설정과 함께 테이블 생성
table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

# Sink 테이블
t_env.create_temporary_table(
    'tumble_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('total_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

print("=== Table API: Tumble Window (5초) ===")
table.window(Tumble.over(lit(5).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), col('price').sum, col("w").start, col("w").end) \
     .execute_insert('tumble_sink') \
     .wait()

In [ ]:
# Sliding Window (Table API)
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'slide_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('total_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

print("=== Table API: Sliding Window (크기: 5초, 슬라이드: 2초) ===")
table.window(Slide.over(lit(5).seconds).every(lit(2).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), col('price').sum, col("w").start, col("w").end) \
     .execute_insert('slide_sink') \
     .wait()

In [ ]:
# Over Window (Table API) - 행 기반 윈도우
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'over_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('max_price', DataTypes.FLOAT())
                           .build())
                   .build()
)

print("=== Table API: Over Window (이전 2행까지의 max) ===")
table.over_window(
    Over.partition_by(col("name"))
        .order_by(col("ts"))
        .preceding(row_interval(2))
        .following(CURRENT_ROW)
        .alias('w')
) \
     .select(col('name'), col('price').max.over(col('w'))) \
     .execute_insert('over_sink') \
     .wait()

### 3.7 Multi-Sink (여러 출력)

`StatementSet`을 사용하면 하나의 소스 데이터를 여러 싱크로 동시에 출력할 수 있습니다.

In [ ]:
# Multi-Sink 예제
t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())

table = t_env.from_elements(
    elements=[(1, 'Hello'), (2, 'World'), (3, "Flink"), (4, "PyFlink")],
    schema=['id', 'data']
)

# 두 개의 Sink 테이블 정의
t_env.execute_sql("""
    CREATE TABLE first_sink (
        id BIGINT,
        data VARCHAR
    ) WITH (
        'connector' = 'print'
    )
""")

t_env.execute_sql("""
    CREATE TABLE second_sink (
        id BIGINT,
        data VARCHAR
    ) WITH (
        'connector' = 'print'
    )
""")

# StatementSet 생성
statement_set = t_env.create_statement_set()

# Sink 1: id <= 3인 데이터
statement_set.add_insert_sql(f"INSERT INTO first_sink SELECT * FROM {table} WHERE id <= 3")

# Sink 2: 'Flink' 포함 데이터
@udf(result_type=DataTypes.BOOLEAN())
def contains_flink(data):
    return "Flink" in data

second_table = table.where(contains_flink(table.data))
statement_set.add_insert("second_sink", second_table)

print("=== Multi-Sink 결과 ===")
print("Sink 1: id <= 3")
print("Sink 2: 'Flink' 포함")
statement_set.execute().wait()

### 3.8 Pandas 연동

In [ ]:
# Pandas DataFrame <-> Flink Table 변환
import pandas as pd
import numpy as np

t_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
t_env.get_config().set("parallelism.default", "1")

# Pandas DataFrame 생성
pdf = pd.DataFrame(np.random.rand(5, 2), columns=['a', 'b'])
print("=== 원본 Pandas DataFrame ===")
print(pdf)

# Pandas -> Flink Table
table = t_env.from_pandas(
    pdf,
    schema=DataTypes.ROW([
        DataTypes.FIELD("a", DataTypes.DOUBLE()),
        DataTypes.FIELD("b", DataTypes.DOUBLE())
    ])
)

# Flink Table -> Pandas
result_pdf = table.to_pandas()
print("\n=== Flink를 거쳐 다시 Pandas로 ===")
print(result_pdf)

In [ ]:
# Pandas UDAF 사용
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(1)
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

ds = env.from_collection(
    collection=window_table_data,
    type_info=Types.ROW([Types.INSTANT(), Types.STRING(), Types.FLOAT()])
)

table = t_env.from_data_stream(
    ds,
    Schema.new_builder()
          .column_by_expression("ts", "CAST(f0 AS TIMESTAMP_LTZ(3))")
          .column("f1", DataTypes.STRING())
          .column("f2", DataTypes.FLOAT())
          .watermark("ts", "ts - INTERVAL '3' SECOND")
          .build()
).alias("ts", "name", "price")

t_env.create_temporary_table(
    'pandas_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('name', DataTypes.STRING())
                           .column('mean_price', DataTypes.FLOAT())
                           .column('w_start', DataTypes.TIMESTAMP_LTZ())
                           .column('w_end', DataTypes.TIMESTAMP_LTZ())
                           .build())
                   .build()
)

# Pandas UDAF: 평균 계산 (Pandas의 mean() 사용)
@udaf(result_type=DataTypes.FLOAT(), func_type="pandas")
def mean_udaf(v):
    return v.mean()

print("=== Pandas UDAF: 윈도우별 평균 ===")
table.window(Tumble.over(lit(5).seconds).on(col("ts")).alias("w")) \
     .group_by(col('name'), col('w')) \
     .select(col('name'), mean_udaf(col('price')), col("w").start, col("w").end) \
     .execute_insert('pandas_sink') \
     .wait()

---
## 4. DataStream과 Table API 혼합 사용

PyFlink에서는 DataStream API와 Table API를 자유롭게 혼합하여 사용할 수 있습니다.

In [45]:
# DataStream <-> Table 변환 예제
env = StreamExecutionEnvironment.get_execution_environment()
t_env = StreamTableEnvironment.create(stream_execution_environment=env)

# 1. Table API로 소스 정의
t_env.create_temporary_table(
    'source',
    TableDescriptor.for_connector('datagen')
                   .schema(Schema.new_builder()
                           .column('id', DataTypes.BIGINT())
                           .column('data', DataTypes.STRING())
                           .build())
                   .option("number-of-rows", "10")
                   .build()
)

# Sink 정의
t_env.create_temporary_table(
    'mix_sink',
    TableDescriptor.for_connector('print')
                   .schema(Schema.new_builder()
                           .column('result', DataTypes.BIGINT())
                           .build())
                   .build()
)

# UDF 정의
@udf(result_type=DataTypes.BIGINT())
def length(data):
    return len(data)

# 2. Table API 연산
table = t_env.from_path("source")
table = table.select(col('id'), length(col('data')))

# 3. Table -> DataStream 변환
ds = t_env.to_data_stream(table)

# 4. DataStream API 연산
ds = ds.map(lambda i: i[0] + i[1], output_type=Types.LONG())

# 5. DataStream -> Table 변환
table = t_env.from_data_stream(ds, col("result"))
print("=== DataStream <-> Table 혼합 사용 ===")
print("1. Table API: datagen 소스")
print("2. Table API: id + length(data) 계산")
print("3. DataStream API: 두 값 합산")
print("4. Table API: 결과 출력")
print()
table.execute_insert('mix_sink').wait()

=== DataStream <-> Table 혼합 사용 ===
1. Table API: datagen 소스
2. Table API: id + length(data) 계산
3. DataStream API: 두 값 합산
4. Table API: 결과 출력



Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
    target=lambda: self._read_inputs(elements_iterator),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 704, in _read_inputs
Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    for elements in elements_iterator:
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 543, in __next__
    return self._next()
           ^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/grpc/_c

8> +I[-9194699132695677926]
4> +I[-1763216453810806078]
3> +I[3388211187165103386]
6> +I[-2403427237402829882]
1> +I[-7610133659905599576]
1> +I[-2306946587656434445]
5> +I[7845795388400597420]
2> +I[-1395032285924183193]
2> +I[-8487444251659080199]


Exception in thread read_grpc_client_inputs:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 721, in <lambda>
    target=lambda: self._read_inputs(elements_iterator),
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/apache_beam/runners/worker/data_plane.py", line 704, in _read_inputs
    for elements in elements_iterator:
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 543, in __next__
    return self._next()
           ^^^^^^^^^^^^
  File "/opt/venv/lib/python3.12/site-packages/grpc/_channel.py", line 969, in _next
    raise self
grpc._channel._MultiThreadedRendezvous: <_MultiThreadedRendezvous of RPC that terminated with:
	status = Sta

7> +I[3990572818205266701]


---
## 5. 커넥터 (Connectors)

PyFlink는 다양한 외부 시스템과의 연동을 위한 커넥터를 제공합니다.

**참고**: 실제 실행을 위해서는 해당 커넥터의 JAR 파일이 필요합니다.

### 5.1 Kafka 커넥터 (예제 코드)

```python
# Kafka JSON Format 예제
from pyflink.datastream.connectors.kafka import (
    KafkaSource, KafkaSink, KafkaRecordSerializationSchema, KafkaOffsetsInitializer
)
from pyflink.datastream.formats.json import JsonRowSerializationSchema, JsonRowDeserializationSchema

# JAR 파일 추가
env.add_jars("file:///path/to/flink-sql-connector-kafka-1.15.0.jar")

# Kafka Source
type_info = Types.ROW([Types.INT(), Types.STRING()])
deserialization_schema = JsonRowDeserializationSchema.Builder() \
    .type_info(type_info) \
    .build()

kafka_source = KafkaSource.builder() \
    .set_topics('my_topic') \
    .set_value_only_deserializer(deserialization_schema) \
    .set_properties({'bootstrap.servers': 'localhost:9092', 'group.id': 'my_group'}) \
    .set_starting_offsets(KafkaOffsetsInitializer.earliest()) \
    .build()

ds = env.from_source(
    kafka_source,
    watermark_strategy=WatermarkStrategy.no_watermarks(),
    source_name="kafka source"
)

# Kafka Sink
serialization_schema = JsonRowSerializationSchema.Builder() \
    .with_type_info(type_info) \
    .build()

record_serializer = KafkaRecordSerializationSchema.builder() \
    .set_topic('output_topic') \
    .set_value_serialization_schema(serialization_schema) \
    .build()

kafka_sink = KafkaSink.builder() \
    .set_record_serializer(record_serializer) \
    .set_bootstrap_servers('localhost:9092') \
    .build()

ds.sink_to(kafka_sink)
```

### 5.2 Elasticsearch 커넥터 (예제 코드)

```python
from pyflink.datastream.connectors.elasticsearch import (
    Elasticsearch7SinkBuilder, FlushBackoffType, ElasticsearchEmitter
)
from pyflink.datastream.connectors import DeliveryGuarantee

# JAR 파일 추가
env.add_jars('file:///path/to/flink-sql-connector-elasticsearch7-1.16.0.jar')

ds = env.from_collection(
    [{'name': 'ada', 'id': '1'}, {'name': 'luna', 'id': '2'}],
    type_info=Types.MAP(Types.STRING(), Types.STRING())
)

# Static Index
es_sink = Elasticsearch7SinkBuilder() \
    .set_emitter(ElasticsearchEmitter.static_index('my_index', 'id')) \
    .set_hosts(['localhost:9200']) \
    .set_delivery_guarantee(DeliveryGuarantee.AT_LEAST_ONCE) \
    .set_bulk_flush_max_actions(1) \
    .build()

# Dynamic Index (데이터 필드 값을 인덱스명으로 사용)
es_sink_dynamic = Elasticsearch7SinkBuilder() \
    .set_emitter(ElasticsearchEmitter.dynamic_index('name', 'id')) \
    .set_hosts(['localhost:9200']) \
    .build()

ds.sink_to(es_sink)
```

### 5.3 Pulsar 커넥터 (예제 코드)

```python
from pyflink.common import SimpleStringSchema
from pyflink.datastream.connectors.pulsar import (
    PulsarSource, PulsarSink, StartCursor, StopCursor, 
    DeliveryGuarantee, TopicRoutingMode
)

# JAR 파일 추가
env.add_jars('file:///path/to/flink-sql-connector-pulsar-1.16.0.jar')

SERVICE_URL = 'pulsar://localhost:6650'
ADMIN_URL = 'http://localhost:8080'

# Pulsar Source
pulsar_source = PulsarSource.builder() \
    .set_service_url(SERVICE_URL) \
    .set_admin_url(ADMIN_URL) \
    .set_topics('input_topic') \
    .set_start_cursor(StartCursor.latest()) \
    .set_unbounded_stop_cursor(StopCursor.never()) \
    .set_subscription_name('my_subscription') \
    .set_deserialization_schema(SimpleStringSchema()) \
    .build()

ds = env.from_source(
    source=pulsar_source,
    watermark_strategy=WatermarkStrategy.for_monotonous_timestamps(),
    source_name="pulsar source"
)

# Pulsar Sink
pulsar_sink = PulsarSink.builder() \
    .set_service_url(SERVICE_URL) \
    .set_admin_url(ADMIN_URL) \
    .set_topics('output_topic') \
    .set_serialization_schema(SimpleStringSchema()) \
    .set_delivery_guarantee(DeliveryGuarantee.AT_LEAST_ONCE) \
    .set_topic_routing_mode(TopicRoutingMode.ROUND_ROBIN) \
    .build()

ds.sink_to(pulsar_sink)
```

---
## 정리

### DataStream API vs Table API 비교

| 특성 | DataStream API | Table API |
|------|---------------|----------|
| 추상화 수준 | 저수준 | 고수준 |
| 상태 접근 | 직접 제어 가능 | 자동 관리 |
| 타이머 | 직접 등록 가능 | 불가능 |
| SQL 지원 | 불가능 | 가능 |
| 최적화 | 수동 | 자동 |
| 학습 곡선 | 높음 | 낮음 |

### 언제 어떤 API를 사용할까?

- **Table API**: SQL/관계형 연산, 빠른 개발, 자동 최적화가 필요한 경우
- **DataStream API**: 세밀한 상태 제어, 복잡한 이벤트 처리, 타이머 사용이 필요한 경우
- **혼합 사용**: 양쪽의 장점을 모두 활용해야 하는 경우

---
## 참고 자료

- [PyFlink 공식 문서](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/overview/)
- [PyFlink DataStream API](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/datastream_tutorial/)
- [PyFlink Table API](https://nightlies.apache.org/flink/flink-docs-release-2.2/docs/dev/python/table_api_tutorial/)
- [Apache Flink GitHub](https://github.com/apache/flink)